# Example 4: Self-Refinement

This example extends the single-paper illustrations in Examples 1–3 to a multi-paper pipeline, processing all 105 papers from Lu et al. (2023) to extract delay lengths (`Delay_Length_Min` and `Delay_Length_Max` in days). Results are compared against the `Delay_Length_Min` coding from Jansen et al.

**Techniques used:** The pipeline combines RAG (hybrid BM25 + vector retrieval), tool calling, and a ReAct agent—following Example 3—with the addition of a **self-refinement module**:
1. A hybrid retriever queries the full paper and generates an initial response. A hybrid retriever combines both keyword-based (e.g., the BM25 algorithm) and semantic meaning-based retrieval to maximize recall rate.
2. An LLM-based evaluator (`ContextAwareGuidelineEvaluator`) checks whether the retrieved context and response are sufficient and correctly formatted.
3. If evaluation fails, the evaluator's feedback is used to refine the retrieval query for an additional attempt (up to `max_retries`).
4. The final response and feedback are passed to the ReAct agent, which calls `analyze_delay_range` to compute the minimum and maximum delay values.

All prompts and pipeline design were finalized before data extraction, blind to the test papers. The self-refinement pipeline achieved 74.04% accuracy, compared to Jansen et al.'s 68.27%.

## Setup

In [2]:
import nest_asyncio
import pandas as pd
import os

nest_asyncio.apply()

In [14]:
#  setup LLMs and embeddings
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# load api key
import openai
import os
from llama_index.llms.openai import OpenAI

with open("./api_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ["OPENAI_API_KEY"] = api_key
openai.api_key = os.getenv("OPENAI_API_KEY")

Settings.llm = OpenAI(model="gpt-4o", api_key=os.environ["OPENAI_API_KEY"])

# specify embedding model

LOCAL_EMBED_PATH = "../models/Qwen3-Embedding-0.6B"
Settings.embed_model = HuggingFaceEmbedding(
    model_name="Qwen/Qwen3-Embedding-0.6B", cache_folder=LOCAL_EMBED_PATH
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1269.37it/s, Materializing param=norm.weight]                              


## File path and papers

In [3]:
# load papers
coding_data = pd.read_excel("../data/coding_data/combined_coding_0224.xlsx")
coding_data.head()

,No.effect size,No.article,singleES,First_author,Publication_year,Matched_Filename,Study_design,Mean_age_total,gpt4_Mean_age_total_clean,Mean_age_total_QW,DD_model,Delay_Length_Min,SS_delay_or_Not,gpt4_Delay_Length_min_clean,gpt4_SS_delay_or_Not_clean,gpt4_Cohort_clean,gpt4_Study_design_clean,gpt4_DD_Model_clean
0,1,5,1,van den Bos et al.,2015,van den Bos et al_2015_Adolescent impatience d...,cross-sectional,16.76,16.76,16.76,a hyperbolic function,15.0,no delay,14,mixed,1994,cross-sectional,a hyperbolic function
1,2,6,1,Olson et al.,2007,Olson et al_2007_Adolescents_ performance on d...,cross-sectional,16.49,16.49,16.49,AUC,1.0,no delay,1,no delay,1990,cross-sectional,AUC
2,3,7,1,Lee et al.,2013,Lee et al_2013_Age and educational track influ...,cross-sectional,14.39,14.39,NaN,AUC,7.0,no delay,7,no delay,1999,cross-sectional,AUC
3,4,9,1,Steinberg et al.,2009,Steinberg et al_2009_Age differences in future...,cross-sectional,NaN,NO,17.91,a hyperbolic function,1.0,no delay,1,no delay,1978,cross-sectional,"a hyperbolic function, an exponential function"
4,5,10,1,Löckenhoff et al.,2020,Löckenhoff_Samanez-Larkin_2020_Age differences...,cross-sectional,48.70,48.7,NaN,an exponential function,NaN,no delay,Study 1: 30,NO,1971,cross-sectional,NO


In [15]:
# load OCR-extracted papers
from llama_index.core import SimpleDirectoryReader
from llama_index.core import SummaryIndex, VectorStoreIndex
import os

# define storage path for caching embeddings of individual papers

# for markdown files
data_dir = "./data/lu_2023_OCR_Mistral"
# for vector index
persist_dir_sample = "./data/storage_VI"

## Define the evaluator

In [16]:
# define the evaluator
# In this section, we defined the guideline evaluator
from llama_index.core.evaluation import GuidelineEvaluator, EvaluationResult
from llama_index.core import PromptTemplate, Response
from typing import Any, Optional, Sequence
from llama_index.core.indices.query.query_transform.feedback_transform import (
    FeedbackQueryTransformation,
)
from llama_index.core.query_engine import RetryGuidelineQueryEngine
import re

# guideline evaluator for evaluating the initial LLM response
delay_min_guideline = """
The response passes (returns YES) IF AND ONLY IF both conditions are met:

1. Context Check: The retrieved context contains all delay values in the delay discounting task.
2. Response Check: The response accurately lists ALL delay values found in the context and converts them to days.

Verdict:
- Return YES only if both #1 and #2 are true.
- Return NO if the context is missing the data OR if the response fails to list them.
"""

# the overall evaluation prompt, the "delay_min_guideline" will be integrated here
CONTEXT_AWARE_EVAL_TEMPLATE = PromptTemplate(
    "You are an evaluator assessing a query-response pair using provided context.\n"
    "Query: {query}\n"
    "Context Retrieved: {context_str}\n"
    "Response Generated: {response}\n"
    "-------------------\n"
    "Guidelines:\n"
    "{guidelines}\n"
    "-------------------\n"
    "Check each numbered condition in the guidelines above.\n"
    "Then output ONLY a JSON object with exactly these two fields:\n"
    '  - "passing": true if ALL conditions are met, false otherwise\n'
    '  - "feedback": one or two sentences identifying which specific condition '
    "failed and what information is missing or incorrect. "
    "If passing, write 'All conditions met.'\n\n"
    "Output ONLY the JSON object, no other text:\n"
    '{"passing": <true|false>, "feedback": "<explanation>"}'
)


# in this part, we customized a evaluator that process both initial responses
# and the retrieved relevant information from the full paper (context)
# based on the response and context, the feedback is generated
class ContextAwareGuidelineEvaluator(GuidelineEvaluator):
    def __init__(self, guidelines: str, **kwargs):
        super().__init__(
            guidelines=guidelines, eval_template=CONTEXT_AWARE_EVAL_TEMPLATE, **kwargs
        )
        self._llm = Settings.llm

    async def aevaluate(
        self,
        query: Optional[str] = None,
        response: Optional[str] = None,
        contexts: Optional[Sequence[str]] = None,
        **kwargs: Any,
    ) -> EvaluationResult:
        """
        Overridden to clean the LLM output before parsing.
        Uses synchronous predict() to avoid Python 3.14 + anyio/httpx weakref
        bug in AsyncClient.__aexit__ (TypeError: cannot create weak reference
        to 'NoneType' object via httpcore AsyncShieldCancellation).
        """
        context_str = "\n".join(contexts) if contexts else ""

        # 1. Get raw string from LLM (sync path avoids anyio CancelScope crash)
        eval_response = self._llm.predict(
            self._eval_template,
            query=query,
            response=response,
            context_str=context_str,
            guidelines=self._guidelines,
        )

        # 2. --- CLEANING LOGIC ---
        cleaned_response = eval_response.strip()

        # Remove Markdown code blocks (```json ... ```)
        if "```" in cleaned_response:
            match = re.search(
                r"```(?:json)?\s*(.*?)\s*```", cleaned_response, re.DOTALL
            )
            if match:
                cleaned_response = match.group(1)

        # Remove double curly braces {{ ... }}
        if cleaned_response.startswith("{{") and cleaned_response.endswith("}}"):
            cleaned_response = cleaned_response[1:-1]

        # 3. Parse
        try:
            eval_data = self._output_parser.parse(cleaned_response)
        except Exception as e:
            # Fallback: if JSON fails completely, assume FAIL to be safe
            print(f"JSON Parsing failed on: {cleaned_response}")
            return EvaluationResult(
                query=query,
                response=response,
                contexts=contexts,
                passing=False,
                score=0.0,
                feedback=f"System Error: Invalid JSON from LLM. Raw: {cleaned_response}",
            )

        return EvaluationResult(
            query=query,
            response=response,
            contexts=contexts,
            passing=eval_data.passing,
            score=1.0 if eval_data.passing else 0.0,
            feedback=eval_data.feedback,
        )

In [17]:
from llama_index.core.prompts import ChatPromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole  # needed by QA_CHAT_TEMPLATE
from llama_index.core import (
    StorageContext,
    load_index_from_storage,
)  # needed by BatchModeratorCoder
from llama_index.core.tools import FunctionTool  # needed by BatchModeratorCoder + agent
from llama_index.core.agent.workflow import AgentStream, ReActAgent, ToolCallResult
from pydantic import BaseModel, Field
from typing import Optional
import asyncio  # needed to wrap async tool
import json

In [18]:
RESEARCHER_SYSTEM_PROMPT = (
    "You are a senior psychology researcher extracting methodological details "
    "from psychology papers. Follow the user's instructions carefully and "
    "output only what is requested."
)

# {context_str} and {query_str} are the exact placeholder names LlamaIndex injects.
QA_CHAT_TEMPLATE = ChatPromptTemplate(
    message_templates=[
        ChatMessage(
            role=MessageRole.SYSTEM,
            content=RESEARCHER_SYSTEM_PROMPT,
        ),
        ChatMessage(
            role=MessageRole.USER,
            content=(
                "Below is the relevant context retrieved from the paper:\n"
                "---------------------\n"
                "{context_str}\n"
                "---------------------\n"
                "Using only the context above, answer the following:\n"
                "{query_str}"
            ),
        ),
    ],
)

In [19]:
import re
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core import get_response_synthesizer
from rank_bm25 import BM25Okapi

from tqdm import tqdm  # for progress bars in batch processing


class DelayRange(BaseModel):
    Delay_Length_Min: Optional[float] = Field(
        default=None, description="Minimum delay in days"
    )
    Delay_Length_Max: Optional[float] = Field(
        default=None, description="Maximum delay in days"
    )


# Short, keyword-dense retrieval query used on attempt 0 to improve
# recall of tables whose headings are semantically distant from the query.
_RETRIEVAL_QUERY = "delay time length future days"

_QUERY_REFINEMENT_TEMPLATE = PromptTemplate(
    "You are improving a retrieval query based on evaluator feedback.\n\n"
    "ORIGINAL QUERY:\n{original_query}\n\n"
    "PRIOR RESPONSE (partial findings already obtained):\n{prior_response}\n\n"
    "EVALUATOR FEEDBACK:\n{feedback}\n\n"
    "SUCCESS CRITERIA (what a passing response must satisfy):\n{guideline}\n\n"
    "---\n"
    "Write an improved version of the original query that:\n"
    "1. Directly targets the specific gap identified by the evaluator.\n"
    "2. Acknowledges correct partial findings from the prior response that "
    "should be preserved or extended.\n"
    "3. Is concise and actionable — do NOT simply append the feedback verbatim.\n\n"
    "Improved query:"
)


class SimpleBM25Retriever(BaseRetriever):
    """Lightweight BM25 retriever using rank-bm25 (no C extensions required)."""

    def __init__(self, nodes, similarity_top_k: int = 3):
        self._nodes = nodes
        self._similarity_top_k = similarity_top_k
        tokenized = [n.get_content().lower().split() for n in nodes]
        self._bm25 = BM25Okapi(tokenized)
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> list[NodeWithScore]:
        tokens = query_bundle.query_str.lower().split()
        scores = self._bm25.get_scores(tokens)
        top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[
            : self._similarity_top_k
        ]
        return [
            NodeWithScore(node=self._nodes[i], score=float(scores[i])) for i in top_n
        ]


def truncate_before_references(text: str) -> str:
    """
    Remove the References section and everything after it.

    Strategy (in order):
    1. Find the first markdown header whose text is exactly 'References'
       (handles # / ## / ### and case-insensitivity).
    2. If not found, fall back to the first 'Discussion' header -- but only
       when there is no Method/Participants/Results section after that point,
       confirming the paper follows the standard IMRaD order.
    3. If neither is found, return the full text unchanged.
    """
    ref_pattern = re.compile(r"^#{1,4}\s+References\s*$", re.MULTILINE | re.IGNORECASE)
    match = ref_pattern.search(text)
    if match:
        return text[: match.start()].rstrip()

    disc_pattern = re.compile(r"^#{1,4}\s+Discussion\s*$", re.MULTILINE | re.IGNORECASE)
    match = disc_pattern.search(text)
    if match:
        after = text[match.start() :]
        method_pattern = re.compile(
            r"^#{1,4}\s+(Method|Participants|Results)\b", re.MULTILINE | re.IGNORECASE
        )
        if not method_pattern.search(after):
            return text[: match.start()].rstrip()

    return text


class BatchModeratorCoder:
    def __init__(
        self,
        data_dir: str,
        persist_dir: str,
        query: str,
        guideline: str,
        max_retries: int = 2,
    ):
        self.llm = Settings.llm
        self.embed_model = Settings.embed_model
        self.data_dir = data_dir
        self.persist_dir = persist_dir
        self.query = query
        self.guideline = guideline
        self.max_retries = max_retries
        self.common_tools = self._setup_function_tools()
        self.PERSIST_DIR_256 = "../data/storage_VI_256"

    def _setup_function_tools(self):

        def analyze_delay_range(values: list) -> dict:
            """
            Compute min and max delay length (days) in the delay discounting task.
            Input: a list of numbers (e.g., [1, 7, 30]).
            Output: {"Delay_Length_Min": <min>, "Delay_Length_Max": <max>}.
            """
            if not values:
                return {"Delay_Length_Min": None, "Delay_Length_Max": None}
            return {"Delay_Length_Min": min(values), "Delay_Length_Max": max(values)}

        return [
            FunctionTool.from_defaults(
                fn=analyze_delay_range,
                name="analyze_delay_range",
                description="Compute min and max delay lengths (in days) from a list of numeric values.",
                # return_direct removed — agent may or may not call this tool
            ),
        ]

    def _refine_query(
        self, original_query: str, prior_response: str, feedback: str
    ) -> str:
        """Use the LLM to diagnose a failed evaluation and polish the original query."""
        return self.llm.predict(
            _QUERY_REFINEMENT_TEMPLATE,
            original_query=original_query,
            prior_response=prior_response,
            feedback=feedback,
            guideline=self.guideline,
        )

    def get_or_create_index(self, file_name: str) -> tuple:
        """Build or load a 256-chunk, pre-screened index. Returns (index, nodes)."""
        file_persist_path = os.path.join(self.PERSIST_DIR_256, file_name)
        if os.path.exists(file_persist_path):
            print(f"  Found 256-chunk index: {file_name}")
            storage_context = StorageContext.from_defaults(
                persist_dir=file_persist_path
            )
            vector_index = load_index_from_storage(
                storage_context, embed_model=self.embed_model
            )
            nodes = list(vector_index.docstore.docs.values())
            return vector_index, nodes

        file_path = os.path.join(self.data_dir, file_name + ".md")
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Paper not found: {file_path}")

        raw_text = open(file_path, encoding="utf-8").read()
        screened_text = truncate_before_references(raw_text)
        print(f"  Pre-screen: {len(raw_text)} -> {len(screened_text)} chars")

        doc = Document(text=screened_text, metadata={"file_name": file_name})
        splitter = SentenceSplitter(chunk_size=256, chunk_overlap=32)
        nodes = splitter.get_nodes_from_documents([doc])
        index = VectorStoreIndex(nodes, embed_model=self.embed_model)
        index.storage_context.persist(persist_dir=file_persist_path)
        print(f"  Built 256-chunk index ({len(nodes)} nodes): {file_name}")
        return index, nodes

    async def _async_refine_retrieval(self, file_name: str) -> tuple[str, list[dict]]:
        """BM25+vector hybrid retrieval with self-refinement loop."""
        try:
            vector_index, nodes = self.get_or_create_index(file_name)
        except FileNotFoundError as e:
            print(f"  SKIPPED: {e}")
            return "", []

        bm25_retriever = SimpleBM25Retriever(nodes=nodes, similarity_top_k=3)
        vector_retriever = vector_index.as_retriever(similarity_top_k=5)
        hybrid_retriever = QueryFusionRetriever(
            [bm25_retriever, vector_retriever],
            num_queries=1,  # no LLM query expansion -- just fuse
            mode="reciprocal_rerank",
            similarity_top_k=6,
        )
        synthesizer = get_response_synthesizer(text_qa_template=QA_CHAT_TEMPLATE)
        evaluator = ContextAwareGuidelineEvaluator(
            guidelines=self.guideline, llm=self.llm
        )

        current_query = self.query
        last_response = "No response generated."
        trace_logs = []

        for attempt in range(self.max_retries):
            print(f"  Attempt {attempt + 1}/{self.max_retries}")
            retrieval_q = _RETRIEVAL_QUERY if attempt == 0 else current_query
            nodes_retrieved = hybrid_retriever.retrieve(retrieval_q)
            print(f"  Nodes: {len(nodes_retrieved)} hybrid nodes")
            response = synthesizer.synthesize(current_query, nodes=nodes_retrieved)
            typed_response = (
                response if isinstance(response, Response) else response.get_response()
            )
            context_list = [n.get_content() for n in nodes_retrieved]
            eval_result = await evaluator.aevaluate(
                query=current_query,
                response=typed_response.response,
                contexts=context_list,
            )

            last_response = typed_response.response

            trace_logs.append(
                {
                    "iteration": attempt + 1,
                    "query_used": current_query,
                    "retrieval_query": retrieval_q,
                    "response": typed_response.response,
                    "contexts": "\n".join(context_list),
                    "feedback": eval_result.feedback,
                    "passing": eval_result.passing,
                }
            )

            if eval_result.passing:
                print("  Passed evaluation.")
                break
            print("  Refining query...")

            current_query = self._refine_query(
                original_query=self.query,
                prior_response=last_response,
                feedback=eval_result.feedback,
            )

        return last_response, trace_logs

    def process_single_paper(self, file_name: str) -> tuple[str, list[dict]]:
        """Full pipeline: self-refinement as a tool + ReActAgent → JSON answer."""
        print(f"\n=== Processing: {file_name} ===")
        trace_logs = []

        def retrieve_paper_details() -> str:
            """Retrieve and refine delay information from a psychology paper."""
            result_text, logs = asyncio.get_event_loop().run_until_complete(
                self._async_refine_retrieval(file_name)
            )
            trace_logs.extend(logs)
            last_feedback = logs[-1]["feedback"] if logs else "No feedback."
            return (
                f"Here is the extracted information and some comments on it:\n"
                f"Response: {result_text}\n"
                f"Feedback: {last_feedback}"
            )

        paper_tool = FunctionTool.from_defaults(
            fn=retrieve_paper_details,
            name="retrieve_paper_details",
            description="Useful for extracting delay information from a research paper.",
        )

        agent_context = """
        You are an expert psychology research assistant.

        Goal:
        Identify ALL delay lengths associated with the larger–later reward in a delay discounting task,
        then determine the minimum and maximum delay lengths (in days).

        Procedure:
        1. Call `retrieve_paper_details` EXACTLY ONCE to obtain delay-related information.
        2. From the response, extract ALL delay lengths for the larger–later reward.
        3. Determine how to compute the delay range:
        - If a complete list of larger–later delays is available, call
            `analyze_delay_range(delays_in_days)`.
        - If delays are given as smaller–sooner delays plus an additional delay for the larger–later reward,
            sum them to obtain larger–later delays, then call `analyze_delay_range`.
        - If no usable delay information is available, call `analyze_delay_range([])` so it returns null values.
        - If the paper explicitly reports only the minimum and/or maximum delay (without the full list),
            do NOT call `analyze_delay_range`; instead, directly report those values.
        4. Strictly follow this output format. Output ONLY the final answer in the following JSON format:
        {"Delay_Length_Min": min_value_or_null, "Delay_Length_Max": max_value_or_null}
        """

        agent = ReActAgent(
            tools=[paper_tool] + self.common_tools,
            llm=self.llm,
            system_prompt=agent_context,
            verbose=True,
        )

        query = "Extract the minimum and maximum delay lengths associated with the larger–later reward in the delay discounting task."

        async def _run_agent():
            handler = agent.run(query, max_iterations=5)
            captured_result = None  # set if analyze_delay_range was called
            try:
                async for ev in handler.stream_events():
                    if isinstance(ev, AgentStream):
                        print(ev.delta, end="", flush=True)
                    elif (
                        isinstance(ev, ToolCallResult)
                        and ev.tool_name == "analyze_delay_range"
                    ):
                        captured_result = (
                            ev.tool_output.content
                        )  # JSON string from tool
            except asyncio.CancelledError:
                pass
            agent_text = None
            try:
                agent_text = await handler  # plain-text response if tool was skipped
            except (asyncio.CancelledError, Exception):
                pass
            return captured_result, str(agent_text) if agent_text else ""

        tool_result, agent_text = asyncio.get_event_loop().run_until_complete(
            _run_agent()
        )

        if tool_result is not None:
            print("  [analyze_delay_range called — using tool output]")
            try:
                data = json.loads(tool_result)
                result_str = DelayRange(**data).model_dump_json()
            except Exception:
                result_str = tool_result  # return raw string on parse failure
        else:
            print("  [tool not called — returning agent plain text]")
            result_str = agent_text

        return result_str, trace_logs

    def run_batch(self, file_list: list) -> pd.DataFrame:
        """Batch-process a list of paper filenames (no file extension)."""
        results = []
        for file_name in tqdm(file_list):
            result_text, logs = self.process_single_paper(file_name)
            results.append(
                {
                    "file_name": file_name,
                    "agent_response": result_text,
                    "trace_logs": logs,
                }
            )
        return pd.DataFrame(results)

In [20]:
FIXED_QUERY = """
Identify the delay lengths associated with the larger–later reward in the delay discounting task
described in this paper.

In one or two sentences:
1. List ALL delay lengths for the larger–later reward in the delay discounting task.
   - If delays are reported as smaller–sooner delays plus an additional delay for the larger–later reward,
     combine them to derive the larger–later delays.
2. Convert EVERY delay length into days (1 week = 7 days, 1 month = 30 days, 1 year = 365 days).
3. If the paper does not explicitly provide sufficient information to determine the delay lengths,
   clearly state that the delay information is not reported.
"""

delay_min_guideline = """
The response passes (returns YES) IF AND ONLY IF both conditions below are satisfied:

1. Context Sufficiency:
   The retrieved context contains sufficient information to determine delay lengths
   for the larger–later reward in the delay discounting task, defined as one of the following:
   - A complete list of delay values for the larger–later reward, OR
   - A complete list of smaller–sooner delays plus an explicit additional delay for the larger–later reward,
     allowing derivation of larger–later delays, OR
   - An explicit report of the minimum and/or maximum delay for the larger–later reward.

2. Response Accuracy:
   The response correctly identifies ALL available delay information for the larger–later reward
   from the context and converts all values into days.

Verdict:
- Return YES only if BOTH Context Sufficiency and Response Accuracy are met.
- Return NO if the context lacks the required information or if the response is incomplete,
  incorrect, or fails to convert delays into days.
"""

In [22]:
# single-paper demo
coder = BatchModeratorCoder(
    data_dir="../data/lu_2023_OCR_Mistral",
    persist_dir="../data/storage_VI",
    query=FIXED_QUERY,
    guideline=delay_min_guideline,
    max_retries=2,
)

# sample_file = "Löckenhoff_Samanez-Larkin_2020_Age differences in intertemporal choice"
# response_text, trace = coder.process_single_paper(sample_file)

# print("\n=== Agent Final Answer (JSON) ===")
# print(response_text)
# print("\n=== Refinement trace (last attempt) ===")
# print(trace[-1] if trace else "No trace.")

## Batch Processing 

In [10]:
file_list = coding_data["Matched_Filename"].tolist()
# remove md suffix
file_list = [f.replace(".md", "") for f in file_list]

In [11]:
len(file_list)

73

In [12]:
coding_data_full = os.listdir("../data/lu_2023_OCR_Mistral")
coding_data_full = [f.replace(".md", "") for f in coding_data_full]

In [13]:
len(coding_data_full)

105

In [23]:
# Batch demo — process multiple papers
df_results = coder.run_batch(coding_data_full)
df_results

  0%|          | 0/105 [00:00<?, ?it/s]


=== Processing: Achterberg et al_2016_Frontostriatal white matter integrity predicts development of delay of ===
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: retrieve_paper_details
Action Input: {}  Found 256-chunk index: Achterberg et al_2016_Frontostriatal white matter integrity predicts development of delay of
  Attempt 1/2
  Nodes: 6 hybrid nodes
  Passed evaluation.
Thought: I have the delay lengths in days. Now, I need to calculate the minimum and maximum delay lengths from the provided values.
Action: analyze_delay_range
Action Input: {"values": [2, 30, 180, 365]}Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The minimum delay length associated with the larger–later reward in the delay discounting task is 2 days, and the maximum delay length is 365 days.

  1%|          | 1/105 [00:11<20:08, 11.62s/it]

  [analyze_delay_range called — using tool output]

=== Processing: Adams_Nettle_2009_Time perspective, personality and smoking, body mass, and physical activity ===
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: retrieve_paper_details
Action Input: {}  Found 256-chunk index: Adams_Nettle_2009_Time perspective, personality and smoking, body mass, and physical activity
  Attempt 1/2
  Nodes: 6 hybrid nodes
  Passed evaluation.
Thought: I have the delay lengths in days. Now, I need to calculate the minimum and maximum delay lengths from these values.
Action: analyze_delay_range
Action Input: {"values": [30, 180, 365, 1825, 3650]}Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The minimum delay length associated with the larger–later reward in the delay discounting task is 30 days, and the maximum delay length is 3,650 days.

  2%|▏         | 2/105 [00:20<16:48,  9.79s/it]

...

  [analyze_delay_range called — using tool output]

=== Processing: Zilker_Pachur_2020_Does option complexity contribute to the framing effect, loss aversion, and ===
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: retrieve_paper_details
Action Input: {}  Found 256-chunk index: Zilker_Pachur_2020_Does option complexity contribute to the framing effect, loss aversion, and
  Attempt 1/2
  Nodes: 6 hybrid nodes
  Passed evaluation.
Thought: I have the delay lengths for the larger–later reward. Now, I need to calculate the minimum and maximum delay lengths from the provided values.
Action: analyze_delay_range
Action Input: {"values": [14, 28, 42]}Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The minimum delay length associated with the larger–later reward is 14 days, and the maximum delay length is 42 days.

100%|██████████| 105/105 [22:47<00:00, 13.63s/it]

  [analyze_delay_range called — using tool output]


100%|██████████| 105/105 [22:47<00:00, 13.03s/it]


,file_name,agent_response,trace_logs
0,Achterberg et al_2016_Frontostriatal white mat...,"{'Delay_Length_Min': 2, 'Delay_Length_Max': 365}","[{'iteration': 1, 'query_used': ' Identify the..."
1,"Adams_Nettle_2009_Time perspective, personalit...","{'Delay_Length_Min': 30, 'Delay_Length_Max': 3...","[{'iteration': 1, 'query_used': ' Identify the..."
2,Alessi_Petry_2003_Pathological gambling severi...,"{'Delay_Length_Min': 0.25, 'Delay_Length_Max':...","[{'iteration': 1, 'query_used': ' Identify the..."
3,Amlung_MacKillop_2014_Clarifying the relations...,The paper does not provide explicit informatio...,"[{'iteration': 1, 'query_used': ' Identify the..."
4,Anandakumar et al_2018_Individual differences ...,"{'Delay_Length_Min': 1, 'Delay_Length_Max': 180}","[{'iteration': 1, 'query_used': ' Identify the..."
...,...,...,...
100,Westwater et al_2019_Combined effects of age a...,The paper does not provide enough information ...,"[{'iteration': 1, 'query_used': ' Identify the..."
101,Whelan 和 McHugh - 2009 - Temporal Discounting ...,"{'Delay_Length_Min': 1, 'Delay_Length_Max': 365}","[{'iteration': 1, 'query_used': ' Identify the..."
102,Wolfe and Patel - 2017 - Instant gratification...,The paper does not report specific delay lengt...,"[{'iteration': 1, 'query_used': ' Identify the..."
103,Yu et al_2021_The neurodevelopment of delay di...,"{'Delay_Length_Min': 2, 'Delay_Length_Max': 180}","[{'iteration': 1, 'query_used': ' Identify the..."


In [24]:
# save the result
df_results.to_csv("../data/coding_data/batch_results_4o.csv", index=False)

## Evaluation

In [3]:
# manually inspect the results and evaluate the performance
df_results_verified = pd.read_csv("../data/coding_data/combined_delay_coding.csv")
df_results_verified.head()

,file_name,First_author,Publication_year,Delay_Length_Min_human,Delay_Length_Min_GPT4o,Delay_Length_Min_GPT4_Jansen,evaluate_RAG,evaluate_Jansen,Notes
0,Achterberg et al_2016_Frontostriatal white mat...,"Achterberg et al., a1",2016.0,10.00,2.00,2.0,0.0,0.0,NaN
1,"Adams_Nettle_2009_Time perspective, personalit...",Adams et al.,2009.0,30.00,30.00,30.0,1.0,1.0,NaN
2,Alessi_Petry_2003_Pathological gambling severi...,Alessi et al.,2003.0,0.25,0.25,25.0,1.0,0.0,NaN
3,Amlung_MacKillop_2014_Clarifying the relations...,Amlung et al.,2014.0,NaN,NaN,NaN,1.0,1.0,NaN
4,Anandakumar et al_2018_Individual differences ...,"Anandakumar et al., a1",2018.0,1.00,1.00,1.0,1.0,1.0,NaN


In [5]:
print("Performance (RAG):")
print(df_results_verified.evaluate_RAG.mean())
print("Performance (naive GPT4o):")
print(df_results_verified.evaluate_Jansen.mean())

Performance (RAG):
0.7403846153846154
Performance (naive GPT4o):
0.6826923076923077
